In [ ]:
####### roomhubs crwaling #########

In [40]:
import re
import time
import random
import hashlib
from datetime import datetime, timezone, timedelta
from urllib.parse import urlparse, urljoin, parse_qs, urlencode, urlunparse

import requests
import pandas as pd
from bs4 import BeautifulSoup

In [48]:
# 1) config + session
BASE_URL = "https://roomhubs.com"
START_URL = "https://roomhubs.com/%ec%84%9c%ec%9a%b8%ec%a7%80%ec%97%ad/"

SOURCE_SITE = urlparse(BASE_URL).netloc
KST = timezone(timedelta(hours=9))

session = requests.Session()
session.headers.update({
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,*/*;q=0.8",
    "Accept-Language": "ko-KR,ko;q=0.9,en;q=0.7",
    "Connection": "keep-alive",
    "Referer": BASE_URL + "/",
})
_ = session.get(BASE_URL + "/", timeout=20)

In [49]:
# 2) utils
def now_kst():
    return datetime.now(KST).strftime("%Y-%m-%d %H:%M:%S%z")

def clean_text(x):
    return re.sub(r"\s+", " ", (x or "")).strip()

def derive_category(btn_text):
    t = clean_text(btn_text)
    t = re.sub(r"^서울\s*", "", t)
    return t or None

def derive_name(h4_text):
    t = clean_text(h4_text)
    t = re.sub(r"^\[[^\]]+\]\s*", "", t)  # remove [서울/강남]
    toks = t.split()
    if toks and toks[0] == "강남":        # exception rule
        toks = toks[1:]
    return " ".join(toks) or None

def extract_external_id(detail_url, fallback):
    if detail_url:
        m = re.search(r"(\d+)(?:/)?$", detail_url)
        if m:
            return m.group(1)
        return hashlib.sha1(detail_url.encode("utf-8")).hexdigest()
    return hashlib.sha1((fallback or "").encode("utf-8")).hexdigest()

def build_page_url(base_url, page):
    u = urlparse(base_url)
    qs = parse_qs(u.query)
    qs["page"] = [str(page)]
    return urlunparse((u.scheme, u.netloc, u.path, u.params,
                       urlencode(qs, doseq=True), u.fragment))

In [50]:
# 3) robust_get
def robust_get(url, timeout=20, max_retries=6, base_sleep=1.0):
    last = None
    for i in range(1, max_retries + 1):
        try:
            time.sleep(base_sleep * (0.4 + random.random()))
            r = session.get(url, timeout=timeout)
            r.raise_for_status()
            return r
        except Exception as e:
            last = e
            time.sleep(base_sleep * (2 ** (i - 1)))
    raise last

In [52]:
# 4) parse (묶임형 + 무료광고 section에서 중단)
def is_free_ad_section(sec) -> bool:
    # 구조: <p><strong>무료</strong> 광고</p>
    txt = clean_text(sec.get_text(" ", strip=True))
    if "무료 광고" in txt:
        return True
    st = sec.select_one("p strong")
    if st and clean_text(st.get_text()) == "무료" and "광고" in txt:
        return True
    return False

def parse_listing_page(html, listing_url, crawled_at):
    soup = BeautifulSoup(html, "html.parser")
    rows = []

    for sec in soup.select("section"):
        # ✅ 무료 광고 section을 만나면 이후는 전부 제외
        if is_free_ad_section(sec):
            break

        btn = sec.select_one("span.elementor-button-text")
        if not btn:
            continue

        btn_text = clean_text(btn.get_text(" ", strip=True))
        if not btn_text.startswith("서울"):
            continue

        category = derive_category(btn_text)

        # ✅ 같은 section 내부 카드만 (너가 확인한 구조)
        cards = sec.select("[data-locations]")
        if not cards:
            continue

        for it in cards:
            a = it.select_one("a[href]")
            detail_url = urljoin(BASE_URL, a["href"]) if a and a.get("href") else None

            h4 = it.select_one("h4")
            name_raw = derive_name(h4.get_text(" ", strip=True) if h4 else "")

            address_raw = None
            icon = it.select_one("i.mi.place")
            if icon:
                li = icon.find_parent("li")
                if li:
                    address_raw = clean_text(li.get_text(" ", strip=True))

            if not address_raw:
                dl = it.get("data-locations")
                if dl:
                    try:
                        locs = json.loads(dl)
                        if isinstance(locs, list) and locs:
                            address_raw = clean_text(locs[0].get("address"))
                    except Exception:
                        pass

            if not any([name_raw, address_raw, detail_url]):
                continue

            fallback = f"{name_raw}|{address_raw}|{detail_url or ''}"
            external_id = extract_external_id(detail_url, fallback)

            rows.append({
                "source_site": SOURCE_SITE,
                "crawled_at": crawled_at,
                "listing_url": listing_url,
                "detail_url": detail_url,
                "external_id": external_id,
                "name_raw": name_raw,
                "category_raw": category,          # 안정적: 노란 버튼 기반
                "address_raw": address_raw,
                "map_url_raw": detail_url or listing_url,
                "status_raw": None,
            })

    return rows

In [53]:
# 5) crawl
def crawl_roomhubs(start_url, max_pages=50, sleep_sec=1.0):
    crawled_at = now_kst()
    all_rows = []
    seen = set()

    for page in range(1, max_pages + 1):
        url = start_url if page == 1 else build_page_url(start_url, page)

        try:
            res = robust_get(url)
        except Exception as e:
            print(f"[STOP] page={page} error={type(e).__name__}: {e}")
            break

        rows = parse_listing_page(res.text, url, crawled_at)

        if not rows:
            print(f"[STOP] page={page} no rows")
            break

        new = 0
        for r in rows:
            eid = r["external_id"]
            if eid not in seen:
                seen.add(eid)
                new += 1

        all_rows.extend(rows)
        print(f"[PAGE {page}] rows={len(rows)} new={new} unique={len(seen)}")

        if page > 1 and new == 0:
            break

        time.sleep(sleep_sec)

    return pd.DataFrame(all_rows)

In [54]:
# 6) run
df_raw = crawl_roomhubs(START_URL)
df_raw.shape, df_raw.head()

[PAGE 1] rows=39 new=23 unique=23
[PAGE 2] rows=39 new=0 unique=23


((78, 10),
     source_site                crawled_at  \
 0  roomhubs.com  2026-01-28 14:48:49+0900   
 1  roomhubs.com  2026-01-28 14:48:49+0900   
 2  roomhubs.com  2026-01-28 14:48:49+0900   
 3  roomhubs.com  2026-01-28 14:48:49+0900   
 4  roomhubs.com  2026-01-28 14:48:49+0900   
 
                                          listing_url  \
 0  https://roomhubs.com/%ec%84%9c%ec%9a%b8%ec%a7%...   
 1  https://roomhubs.com/%ec%84%9c%ec%9a%b8%ec%a7%...   
 2  https://roomhubs.com/%ec%84%9c%ec%9a%b8%ec%a7%...   
 3  https://roomhubs.com/%ec%84%9c%ec%9a%b8%ec%a7%...   
 4  https://roomhubs.com/%ec%84%9c%ec%9a%b8%ec%a7%...   
 
                                           detail_url  \
 0  https://roomhubs.com/listing/%ec%84%9c%ec%9a%b...   
 1  https://roomhubs.com/listing/%ec%84%9c%ec%9a%b...   
 2  https://roomhubs.com/listing/%ec%a9%9c%ec%98%a...   
 3  https://roomhubs.com/listing/%ea%b0%95%eb%82%a...   
 4  https://roomhubs.com/listing/%ec%84%9c%ec%9a%b...   
 
                       

In [55]:
# 7) merge (external_id 기준, 버리지 않고 병합)
def first_non_null(s):
    for v in s:
        if pd.notna(v) and v not in ["", None]:
            return v
    return None

def join_unique(s):
    vals = [str(v) for v in s.dropna().unique() if str(v).strip()]
    return " | ".join(vals) if vals else None

df_merged = (
    df_raw
    .groupby("external_id", dropna=False, as_index=False)
    .agg({
        "source_site": first_non_null,
        "crawled_at": first_non_null,
        "listing_url": join_unique,
        "detail_url": join_unique,
        "name_raw": first_non_null,
        "category_raw": first_non_null,
        "address_raw": first_non_null,
        "map_url_raw": first_non_null,
        "status_raw": first_non_null,
    })
)

df_merged.shape, df_merged.head()

((23, 10),
                                 external_id   source_site  \
 0                                         0  roomhubs.com   
 1  02fc437f1ef85e37155c9deae28229e0fff52b66  roomhubs.com   
 2  06e42642286fccb401525418a282d9843bdea8b1  roomhubs.com   
 3                                        11  roomhubs.com   
 4  192b9810584e488072d350ae2b3bb2339b814f52  roomhubs.com   
 
                  crawled_at  \
 0  2026-01-28 14:48:49+0900   
 1  2026-01-28 14:48:49+0900   
 2  2026-01-28 14:48:49+0900   
 3  2026-01-28 14:48:49+0900   
 4  2026-01-28 14:48:49+0900   
 
                                          listing_url  \
 0  https://roomhubs.com/%ec%84%9c%ec%9a%b8%ec%a7%...   
 1  https://roomhubs.com/%ec%84%9c%ec%9a%b8%ec%a7%...   
 2  https://roomhubs.com/%ec%84%9c%ec%9a%b8%ec%a7%...   
 3  https://roomhubs.com/%ec%84%9c%ec%9a%b8%ec%a7%...   
 4  https://roomhubs.com/%ec%84%9c%ec%9a%b8%ec%a7%...   
 
                                           detail_url     name_raw  \
 0  htt

In [56]:
# 8) save
import os
os.makedirs("../data", exist_ok=True)

ts = datetime.now(KST).strftime("%Y%m%d_%H%M%S")
out_path = f"../data/roomhubs_seoul_raw_{ts}.csv"

df_merged.to_csv(out_path, index=False, encoding="utf-8-sig")
out_path

'../data/roomhubs_seoul_raw_20260128_144910.csv'